# Lesson 4: Deployment

**Complete notebook.** Every cell runs as it is. Run the cells in order, top to
bottom, and read the explanation between them: what matters is what the output
shows and why.

## What this notebook builds

The trained model from Lesson 3, wrapped so that somebody who is not in this room
can use it. A single page with upload controls, a public link, and a reply that is
structured data rather than a picture. The same page also tracks people across
frames and reports who waited inside a fixed area, and for how long.

Everything runs in Colab. There is no container and no cloud project. What is
being taught is the step from a notebook that works for the person who wrote it
to a thing that survives a stranger, and that step is the same either way: load
the model once, refuse bad input clearly, answer in a shape a caller can parse.

## The order of work

1. Setup and the weights from Lesson 3
2. Load the model once
3. The functions that are the whole service
4. Who is waiting, and for how long, inside a fixed area
5. The interface, and a public link
6. Testing the error conditions
7. Cold and warm response time
8. Cloud or edge

## Timing

The build is short. Most of the lab goes on Section 4, the waiting half, and
Section 6, the error table. Those are the parts the sheet assesses, so do not let
them be the parts that get rushed.


---

## 1. Setup

Gradio builds the page. Ultralytics is the same pinned version as Lesson 3, since
weights and library have to agree. ByteTrack, the tracker the waiting half uses,
needs `lap`, the small library that solves the matching between boxes.


In [268]:
ULTRALYTICS_VERSION = "8.4.120"

try:
    import google.colab  # noqa: F401
    RUNNING_ON_COLAB = True
except ImportError:
    RUNNING_ON_COLAB = False

if RUNNING_ON_COLAB:
    import subprocess
    # gradio is left unpinned, because the page only needs a recent version. The
    # version this prints is the one the page was built with. lap is the small
    # library ByteTrack uses to solve the matching between boxes.
    subprocess.run(
        ["pip", "install", "-q", "ultralytics==" + ULTRALYTICS_VERSION, "gradio",
         "lap"],
        check=True,
    )

print("Running on Colab:", RUNNING_ON_COLAB)


Running on Colab: False


In [269]:
import io
import itertools
import json
import math
import os
import shutil
import subprocess
import tempfile
import time
from collections import Counter

import cv2
import gradio as gr
import numpy as np
from PIL import Image, UnidentifiedImageError
from ultralytics import YOLO

print("gradio:", gr.__version__)

if RUNNING_ON_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = "/content/drive/MyDrive/C389"
else:
    def find_project_root(start):
        current = os.path.abspath(start)
        while True:
            if os.path.isdir(os.path.join(current, "data", "clips")):
                return current
            parent = os.path.dirname(current)
            if parent == current:
                break
            current = parent
        return os.path.abspath(os.path.join(os.getcwd(), ".."))

    PROJECT_ROOT = find_project_root(os.getcwd())

WEIGHTS = os.path.join(PROJECT_ROOT, "artifacts", "L3", "best_1.pt")
print("weights:", WEIGHTS)


gradio: 6.31.0
weights: c:\Republic Polytechnic\Y2\Y2Sem2\C389\artifacts\L3\best_1.pt


### If the weights are missing

`best.pt` is written by the Lesson 3 notebook into `artifacts/L3/`. If it is not
there, the cell below falls back to the stock pretrained model so that the rest of
the lab can still be built and tested.

The fallback model detects people and cannot report direction, so every count
comes back under a single `person` class. That is a working service answering
honestly about a model that was never trained for this job, which is a more useful
thing to look at than an error.

The same choice decides which weights the waiting half uses. A real clip uses
`best.pt`, or the stock model when it is missing. The drawn sample clip uses the
sample weights beside it, because a model trained on photographs finds nobody in
a drawing.


In [270]:
if os.path.exists(WEIGHTS):
    MODEL_PATH = WEIGHTS
    USING_TRAINED_MODEL = True
else:
    MODEL_PATH = "yolo26n.pt"
    USING_TRAINED_MODEL = False
    print("WARNING: no best.pt at %s" % WEIGHTS)
    print("Falling back to the stock model. It cannot report direction.")
    print("Re-run the Lesson 3 notebook to the end before submitting anything.")

---

## 2. Load the model once

Once, here, at startup, and never inside the function that answers a request.

Loading a YOLO model takes seconds. Inside the handler those seconds are added to
every single call, and a second caller arriving while the first is still waiting
loads a second copy into memory. This is the single most common way a working
model becomes an unusable service, and it is one line in the wrong place.

The class list is read from the weights rather than typed here, so a CA1 twist
that appends a class needs no edit in this notebook.


In [271]:
MODEL = YOLO(MODEL_PATH)
CLASS_NAMES = [MODEL.names[key] for key in sorted(MODEL.names)]

CONFIDENCE_THRESHOLD = 0.40
MAX_IMAGE_PIXELS = 40_000_000
SERVICE_VERSION = "2.2.0"

print("model loaded from  :", MODEL_PATH)
print("classes            :", CLASS_NAMES)
print("confidence threshold:", CONFIDENCE_THRESHOLD)


model loaded from  : c:\Republic Polytechnic\Y2\Y2Sem2\C389\artifacts\L3\best_1.pt
classes            : ['person_left', 'person_right']
confidence threshold: 0.4


---

## 3. The functions that are the whole service

Validate the input, count what is in it, build the reply. Everything else on this
page is Gradio, and Gradio is replaceable. These functions are what would move
unchanged into a container on a cloud host or onto a device at the edge.

Keeping them separate from the interface is the reason that claim is true rather
than a thing said in a slide.


In [272]:
def validate_image(uploaded):
    """Check that what arrived can be turned into an image to run on.

    Args:
        uploaded: whatever the caller sent. A path, a file like object, a PIL
            image, or None when nothing was sent at all

    Returns:
        (image, error), exactly one of which is None. The error is a dict
        carrying a status code and a message the caller could act on

    Requirement:
        Four cases are refused before any inference runs: nothing sent, an empty
        file, a file that is not a readable image, and an image so large it would
        exhaust memory.

        A service that hands a text file to the model and returns a stack trace
        has told the caller nothing. Every refusal here names what was wrong and
        what to send instead.
    """
    if uploaded is None:
        return None, {"status": 400, "error": "No image was sent.",
                      "detail": "Attach a JPEG or PNG and send it again."}

    try:
        if isinstance(uploaded, Image.Image):
            image = uploaded
        elif isinstance(uploaded, (bytes, bytearray)):
            if not uploaded:
                return None, {"status": 400, "error": "The upload was empty.",
                              "detail": "The file carried no data."}
            image = Image.open(io.BytesIO(uploaded))
            image.load()
        else:
            if os.path.getsize(uploaded) == 0:
                return None, {"status": 400, "error": "The upload was empty.",
                              "detail": "The file '%s' is zero bytes."
                                        % os.path.basename(str(uploaded))}
            image = Image.open(uploaded)
            image.load()
    except (UnidentifiedImageError, OSError):
        return None, {"status": 415, "error": "That file is not a readable image.",
                      "detail": "Send a JPEG or PNG."}

    if image.width * image.height > MAX_IMAGE_PIXELS:
        return None, {"status": 413, "error": "The image is too large to process.",
                      "detail": "%d by %d pixels is beyond the limit of %d pixels."
                                % (image.width, image.height, MAX_IMAGE_PIXELS)}

    return image.convert("RGB"), None

In [273]:
def count_people(image, threshold):
    """Run the model on one image and total the detections by class.

    Args:
        image: a PIL image, already validated
        threshold: detections below this confidence are dropped

    Returns:
        (counts, kept), where counts maps every class name to a whole number

    Requirement:
        The same rule as Lesson 3. Every class appears in counts, including the
        ones at zero, and an image with nobody in it is a valid result of zero
        rather than an error.
    """
    result = MODEL.predict(image, conf=0.01, verbose=False)[0]

    kept = []
    boxes = result.boxes
    if boxes is not None:
        for index in range(len(boxes)):
            confidence = round(float(boxes.conf[index].item()), 3)
            if confidence < threshold:
                continue
            class_id = int(boxes.cls[index].item())
            kept.append({
                "class_name": CLASS_NAMES[class_id],
                "confidence": confidence,
                "box": [round(float(v), 1) for v in boxes.xyxy[index].tolist()],
            })

    counts = {}
    for name in CLASS_NAMES:
        counts[name] = 0
    for detection in kept:
        counts[detection["class_name"]] = counts[detection["class_name"]] + 1

    return counts, kept

In [274]:
def build_reply(uploaded, threshold=CONFIDENCE_THRESHOLD):
    """The whole service, in one function. Validate, count, answer.

    Returns a dict either way. A caller never has to tell success from failure by
    reading prose: a reply carrying "error" failed, and one carrying "counts" did
    not.

    A CA1 twist that adds a class or an extra field changes this function and
    nothing else, which is why the reply is built in one place.
    """
    image, error = validate_image(uploaded)
    if error is not None:
        return error, None

    started = time.time()
    counts, detections = count_people(image, threshold)
    elapsed = time.time() - started

    reply = {
        "status": 200,
        "counts": counts,
        "confidence_threshold": threshold,
        "model_version": SERVICE_VERSION,
        "trained_model": USING_TRAINED_MODEL,
        "image_size": [image.width, image.height],
        "inference_seconds": round(elapsed, 3),
        "detections": detections,
    }
    # Printed rather than kept, so the Colab output below the cell is this
    # service's log. Section 7 reads it.
    print("counted %s in %.3fs on a %dx%d image"
          % (counts, elapsed, image.width, image.height))
    return reply, image

---

## 4. Who is waiting, and for how long

The photograph half answers one question about one frame. The waiting half
answers a question about time: who stayed inside a marked area, and for how long.

The original demo assumes a fixed camera. If the uploaded camera moves, the
same pixel polygon can drift away from the physical waiting area. The page now
includes an optional stabilisation step before tracking. When enabled, the
system creates a stabilised copy first, then uses that same copy for tracking
and annotation. Draw or recalibrate `WAITING_AREA` on the stabilised output.
Stabilisation reduces camera shake but cannot guarantee correction of strong
perspective changes or parallax.

The polygon below is the original example for the 1280 by 720 demo clip:

```python
WAITING_AREA = [(1000, 450), (1250, 500), (1200, 600), (1000, 600)]
```

The bottom edge of the person box has to land inside the polygon, or within the
edge tolerance. The annotated video draws the area on every frame, so the first
thing to check is whether the people who look like they are waiting are actually
inside the drawn shape.

Three things have to be true before a dwell time means anything:

- **The area is known in every frame.** A fixed camera means a fixed pixel
  polygon points at the same piece of ground in every frame.
- **Every observation is checked against the same area.** The area does not move,
  so there is no per-frame reconstruction to go wrong.
- **Every frame is read in order.** A detector gives boxes per frame and no
  identity. ByteTrack links the boxes across frames, which is what makes "this is
  the same person as one second ago" possible.

The clip is not resized. OpenCV reads it at its recorded size, the tracker sees
the original pixels, and the annotated video is written at that size. Ultralytics
does resize each frame internally to its inference size, but that happens inside
the model and the boxes come back in the original coordinates.

The area was marked on a 1280 by 720 frame. If an uploaded clip is a different
size, the same numbers no longer point at the same place, and the log warns about
the mismatch.


In [ ]:
DEMO_CLIP = os.path.join(PROJECT_ROOT, "data", "clips", "demo1.mp4")
SAMPLE_DIRS = [
    os.path.join(PROJECT_ROOT, "data", "sample", "clip"),
    os.path.join(PROJECT_ROOT, "week3", "clip"),
]

# The demo clip comes from a fixed camera, so the waiting area is a fixed
# polygon in the frame. The bottom of a person box must land inside it.
WAITING_AREA = [(1000, 450), (1250, 500), (1200, 600), (1000, 600)]

CLIP_CONFIDENCE = 0.10
CLIP_IMAGE_SIZE = 960
# BoT-SORT uses Ultralytics' built-in botsort.yaml configuration.
MIN_DWELL = 2.0
MAX_WAIT_SPEED_PX_S = 30.0
WAIT_GAP = 1.0
MAX_CLIP_FRAMES = 3000
MAX_FRAME_PIXELS = 40_000_000
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "artifacts", "L4")
os.makedirs(OUTPUT_DIR, exist_ok=True)
TRACKER = "botsort.yaml"
TRACKER_BUFFER = 30


def create_botsort_config(track_buffer):
    """Create a BoT-SORT config using the selected memory buffer."""
    path = os.path.join(OUTPUT_DIR, "botsort_custom.yaml")
    config = """tracker_type: botsort
track_high_thresh: 0.10
track_low_thresh: 0.03
new_track_thresh: 0.10
track_buffer: %d
match_thresh: 0.80
fuse_score: True
gmc_method: sparseOptFlow
proximity_thresh: 0.5
appearance_thresh: 0.25
with_reid: True
model: auto
""" % int(track_buffer)
    with open(path, "w", encoding="utf-8") as tracker_file:
        tracker_file.write(config)
    return path

# The website can override this value for each video analysis.
EDGE_TOLERANCE = 20
MIN_TRACK_OBSERVATIONS = 3
MIN_TRACK_DURATION_SECONDS = 3.0
CLASS_COLOURS_BGR = {
    "person_left": (255, 0, 0),
    "person_right": (0, 255, 255),
}
CLASS_COLOURS_RGB = {
    "person_left": (0, 0, 255),
    "person_right": (255, 255, 0),
}
WAITING_BORDER_BGR = (255, 0, 255)
WAITING_BORDER_RGB = (255, 0, 255)

SAMPLE_CLIP = None
SAMPLE_WEIGHTS = None
for directory in SAMPLE_DIRS:
    truth_path = os.path.join(directory, "clip.json")
    weights_path = os.path.join(directory, "sample_walkway.pt")
    if not os.path.exists(weights_path):
        continue
    clip_name = "walkway_clip.mp4"
    if os.path.exists(truth_path):
        with open(truth_path, encoding="utf-8") as handle:
            clip_name = json.load(handle).get("file", clip_name)
    candidate = os.path.join(directory, clip_name)
    if not os.path.exists(candidate):
        candidate = os.path.join(directory, "walkway_clip.mp4")
    if os.path.exists(candidate):
        SAMPLE_CLIP = candidate
        SAMPLE_WEIGHTS = weights_path
        break


def clip_model_for(path):
    """Use Lesson 3 weights when available, otherwise use a stock person model."""
    if os.path.exists(WEIGHTS):
        return WEIGHTS
    print("WARNING: trained best.pt not found; using stock yolo26n.pt for clips.")
    print("Copy artifacts/L3/best.pt into this project to evaluate the trained model.")
    return "yolo26n.pt"


if os.path.exists(DEMO_CLIP):
    CLIP_PATH = DEMO_CLIP
elif SAMPLE_CLIP is not None:
    CLIP_PATH = SAMPLE_CLIP
else:
    CLIP_PATH = None

if CLIP_PATH is None:
    CLIP_MODEL_PATH = WEIGHTS if os.path.exists(WEIGHTS) else "yolo26n.pt"
else:
    CLIP_MODEL_PATH = clip_model_for(CLIP_PATH)

print("clip        :", CLIP_PATH)
print("clip model  :", CLIP_MODEL_PATH)
print("waiting area:", WAITING_AREA)
print("edge tolerance:", EDGE_TOLERANCE, "px")
if CLIP_PATH is None:
    print("")
    print("No demo clip and no drawn sample clip. Upload a clip in the page.")
elif SAMPLE_CLIP is not None and CLIP_PATH == SAMPLE_CLIP:
    print("")
    print("WARNING: using the drawn sample clip. Its people and waits were placed")
    print("on purpose. The static waiting area is calibrated for demo1, so the")
    print("dwell times from the sample are not a real measurement.")


clip        : c:\Republic Polytechnic\Y2\Y2Sem2\C389\data\clips\demo1.mp4
clip model  : c:\Republic Polytechnic\Y2\Y2Sem2\C389\artifacts\L3\best_1.pt
waiting area: [(1000, 450), (1250, 500), (1200, 600), (1000, 600)]
edge tolerance: 20 px


waited: 1, crossed: 2, 24.6s


### The clip service

`validate_clip` refuses what cannot be tracked before the model sees it. The
statuses have the same shape as the photograph service: `400` for nothing or an
empty file, `415` for a file OpenCV cannot read, `413` for a clip that is too long
or too large, and `200` for a clip that can be processed.

`track_clip` reads the clip one frame at a time. It creates a fresh model for
every run: the tracker lives inside the model, so reusing one would carry the
previous run's identities into this one. The class names are read from that fresh
model rather than from the photograph model, because the sample weights beside
the drawn clip have different class names from `best.pt`.


When the stabilisation checkbox is enabled, the uploaded clip is processed in
two passes before tracking. The first estimates camera motion using optical
flow and robust affine fitting; the second smooths that motion and writes a
corrected video. YOLO/ByteTrack and the annotated output both use this corrected
video, so their pixel coordinates remain consistent. Disable the checkbox for
a fixed-camera clip.


In [276]:
def inside_area(point, area):
    """True when a point is inside a polygon, using the even-odd rule.

    Args:
        point: (x, y) in pixels
        area: list of (x, y) corners in order

    Returns:
        True when the point is inside the polygon, False otherwise

    Requirement:
        The area is an arbitrary quadrilateral, not an axis-aligned rectangle.
        The feet point is on the ground, which is where the marked corners are.
    """
    x, y = point
    inside = False
    previous = len(area) - 1
    for current in range(len(area)):
        x1, y1 = area[current]
        x2, y2 = area[previous]
        crosses = (y1 > y) != (y2 > y)
        if crosses:
            boundary = (x2 - x1) * (y - y1) / (y2 - y1) + x1
            if x < boundary:
                inside = not inside
        previous = current
    return inside


def point_in_or_near_area(point, area, tolerance=EDGE_TOLERANCE):
    """True when a point is inside the area or within tolerance pixels outside it."""
    contour = np.array(area, dtype=np.float32).reshape(-1, 1, 2)
    signed_distance = cv2.pointPolygonTest(
        contour, (float(point[0]), float(point[1])), True)
    return signed_distance >= -tolerance


def observation_in_area(observation, area, tolerance=EDGE_TOLERANCE):
    """True when any bottom-edge point of the box is inside or near the area.

    The three points are the bottom-left, bottom-centre and bottom-right of the
    box. A person standing on the boundary therefore still counts in.
    """
    left, top, right, bottom = observation["box"]
    bottom_points = [
        (left, bottom),
        ((left + right) / 2.0, bottom),
        (right, bottom),
    ]
    return any(point_in_or_near_area(point, area, tolerance)
               for point in bottom_points)


def validate_clip(uploaded):
    """Check that what arrived can be opened as a video.

    Args:
        uploaded: a path, a Gradio file object, or None

    Returns:
        (clip, error), exactly one of which is None. clip holds the path and the
        metadata every later step needs.

    Requirement:
        The four refusals are the clip equivalents of the photograph ones:
        nothing sent, an empty file, a file OpenCV cannot read, and a clip that
        is too long or too large to process. A zero-waiting result is not an
        error and is not refused here.
    """
    if uploaded is None:
        return None, {"status": 400, "error": "No clip was sent.",
                      "detail": "Attach an MP4 and send it again."}

    if isinstance(uploaded, dict):
        uploaded = (uploaded.get("video") or uploaded.get("path")
                    or uploaded.get("name"))

    if not isinstance(uploaded, (str, os.PathLike)):
        return None, {"status": 400, "error": "The clip was not a file path.",
                      "detail": "Send an MP4 file."}

    path = os.fspath(uploaded)
    if not os.path.exists(path):
        return None, {"status": 400, "error": "The clip was not found.",
                      "detail": "The file '%s' does not exist."
                                % os.path.basename(path)}
    if os.path.getsize(path) == 0:
        return None, {"status": 400, "error": "The upload was empty.",
                      "detail": "The file '%s' is zero bytes."
                                % os.path.basename(path)}

    video = cv2.VideoCapture(path)
    try:
        if not video.isOpened():
            return None, {"status": 415,
                          "error": "That file is not a readable video.",
                          "detail": "Send an MP4 that OpenCV can open."}

        fps = float(video.get(cv2.CAP_PROP_FPS))
        frames = int(video.get(cv2.CAP_PROP_FRAME_COUNT))
        width = int(video.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(video.get(cv2.CAP_PROP_FRAME_HEIGHT))

        if (not math.isfinite(fps) or fps <= 0 or frames <= 0
                or width <= 0 or height <= 0):
            return None, {"status": 415,
                          "error": "The video metadata is not usable.",
                          "detail": "OpenCV could not read the frame size or rate."}

        if frames > MAX_CLIP_FRAMES:
            return None, {"status": 413,
                          "error": "The clip is too long to process.",
                          "detail": "%d frames is beyond the limit of %d."
                                    % (frames, MAX_CLIP_FRAMES)}

        if width * height > MAX_FRAME_PIXELS:
            return None, {"status": 413,
                          "error": "The clip frames are too large.",
                          "detail": "%d by %d pixels is beyond the limit of %d pixels."
                                    % (width, height, MAX_FRAME_PIXELS)}

        ok, frame = video.read()
        if not ok or frame is None:
            return None, {"status": 415,
                          "error": "The first frame could not be read.",
                          "detail": "The file is a video container with no usable frames."}

        clip = {
            "path": path,
            "name": os.path.basename(path),
            "fps": fps,
            "frames": frames,
            "width": width,
            "height": height,
            "duration_seconds": round(frames / fps, 3),
        }
    finally:
        video.release()

    return clip, None


def first_frame_preview(uploaded):
    clip, error = validate_clip(uploaded)
    if error is not None:
        return None, None, None, [], [], error["detail"], error["detail"]
    video = cv2.VideoCapture(clip["path"])
    try:
        ok, frame = video.read()
    finally:
        video.release()
    if not ok or frame is None:
        msg = "The first frame could not be read."
        return None, None, None, [], [], msg, msg
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    return (rgb, rgb.copy(), rgb, [], [],
            "Left area: click 4 points on the frame.",
            "Right area: click 4 points on the frame (optional).")


def draw_roi_preview(image, points, colour):
    if image is None:
        return None
    marked = np.array(image).copy()
    points = [(int(point[0]), int(point[1])) for point in (points or [])]
    for number, point in enumerate(points, 1):
        cv2.circle(marked, point, 7, colour, -1)
        cv2.putText(marked, str(number), (point[0] + 9, point[1] - 9),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 0), 4, cv2.LINE_AA)
        cv2.putText(marked, str(number), (point[0] + 9, point[1] - 9),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 1, cv2.LINE_AA)
    if len(points) >= 2:
        cv2.polylines(marked, [np.array(points, dtype=np.int32)], len(points) == 4, colour, 4)
    return marked


def select_roi_point(original_image, points, selection: gr.SelectData, colour, label):
    points = list(points or [])
    index = getattr(selection, "index", None)
    if not isinstance(index, (list, tuple)) or len(index) < 2:
        return draw_roi_preview(original_image, points, colour), points, "Click directly on the first-frame preview."
    if original_image is None:
        return None, points, "Upload a clip first."
    x, y = float(index[0]), float(index[1])
    height, width = original_image.shape[:2]
    if not (0 <= x < width and 0 <= y < height):
        return draw_roi_preview(original_image, points, colour), points, "The selected point is outside the preview."
    if len(points) >= 4:
        points = []
    points.append([round(x, 1), round(y, 1)])
    status = "%s: selected %d/4 points." % (label, len(points))
    if len(points) == 4:
        status = "%s ready." % label
    return draw_roi_preview(original_image, points, colour), points, status


def select_left_roi(original_image, points, evt: gr.SelectData):
    return select_roi_point(original_image, points, evt, (255, 0, 0), "Left area")


def select_right_roi(original_image, points, evt: gr.SelectData):
    return select_roi_point(original_image, points, evt, (0, 255, 255), "Right area")


def validate_waiting_areas(left, right, width, height, return_source=False):
    selected = {}
    source = {"left": "user", "right": "none"}
    for name, area in (("left", left), ("right", right)):
        if area is None or area == []:
            continue
        if not isinstance(area, (list, tuple)) or len(area) != 4:
            result = (None, "Select exactly four points for the %s waiting area." % name)
            return result if not return_source else (None, result[1], source)
        try:
            points = [(float(point[0]), float(point[1])) for point in area]
        except (TypeError, IndexError, ValueError):
            message = "Each %s waiting-area point needs x and y coordinates." % name
            return (None, message) if not return_source else (None, message, source)
        if any(not (math.isfinite(x) and math.isfinite(y)) for x, y in points):
            message = "%s waiting-area points must be finite." % name
            return (None, message) if not return_source else (None, message, source)
        if any(x < 0 or y < 0 or x >= width or y >= height for x, y in points):
            message = "%s waiting-area points must be inside the preview." % name
            return (None, message) if not return_source else (None, message, source)
        contour = np.array(points, dtype=np.float32).reshape(-1, 1, 2)
        if abs(cv2.contourArea(contour)) < 1.0:
            message = "The %s waiting-area polygon must enclose a visible area." % name
            return (None, message) if not return_source else (None, message, source)
        selected[name] = [[round(x, 1), round(y, 1)] for x, y in points]
        source[name] = "user"
    return (selected, None, source) if return_source else (selected, None)


In [277]:
def stabilise_clip(path, progress=None):
    """Lock every frame to the first frame using global camera motion."""
    video = cv2.VideoCapture(path)
    if not video.isOpened():
        video.release()
        raise RuntimeError("OpenCV could not open the clip for stabilisation.")

    fps = float(video.get(cv2.CAP_PROP_FPS))
    width = int(video.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(video.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(video.get(cv2.CAP_PROP_FRAME_COUNT))
    ok, reference = video.read()
    if not ok or reference is None:
        video.release()
        raise RuntimeError("The first frame could not be read for stabilisation.")

    output_path = os.path.join(OUTPUT_DIR,
                               "stabilised_%s.mp4" % next(tempfile._get_candidate_names()))
    writer = cv2.VideoWriter(
        output_path, cv2.VideoWriter_fourcc(*"mp4v"), fps, (width, height))
    if not writer.isOpened():
        video.release()
        raise RuntimeError("OpenCV could not create the stabilised clip.")

    feature_params = {
        "maxCorners": 500,
        "qualityLevel": 0.01,
        "minDistance": 12,
        "blockSize": 7,
    }
    lk_params = {
        "winSize": (21, 21),
        "maxLevel": 3,
        "criteria": (cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 30, 0.01),
    }
    previous_gray = cv2.cvtColor(reference, cv2.COLOR_BGR2GRAY)
    previous_points = cv2.goodFeaturesToTrack(previous_gray, **feature_params)
    previous_to_reference = np.eye(3, dtype=np.float32)
    processed = 0
    started = time.time()
    writer.write(reference)
    report_progress(progress, "Stabilising", 1, total_frames, started)

    try:
        while True:
            ok, frame = video.read()
            if not ok:
                break
            processed += 1
            current_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            transform = None
            if previous_points is not None and len(previous_points) >= 6:
                tracked_points, status, _ = cv2.calcOpticalFlowPyrLK(
                    previous_gray, current_gray, previous_points, None, **lk_params)
                if tracked_points is not None and status is not None:
                    valid_previous = previous_points[status.ravel() == 1]
                    valid_current = tracked_points[status.ravel() == 1]
                    if len(valid_previous) >= 6:
                        transform, inliers = cv2.estimateAffinePartial2D(
                            valid_previous, valid_current, method=cv2.RANSAC,
                            ransacReprojThreshold=3.0, maxIters=2000, confidence=0.99)
                        if transform is not None and inliers is not None:
                            if int(inliers.sum()) < 6:
                                transform = None
                        previous_points = valid_current.reshape(-1, 1, 2)
            if transform is not None:
                try:
                    current_to_previous = cv2.invertAffineTransform(transform)
                    current_to_previous = np.vstack((current_to_previous, [0, 0, 1]))
                    previous_to_reference = (previous_to_reference @ current_to_previous)
                except (cv2.error, ValueError):
                    transform = None
            if transform is None:
                previous_points = cv2.goodFeaturesToTrack(current_gray, **feature_params)
            stabilised = cv2.warpAffine(
                frame, previous_to_reference[:2].astype(np.float32), (width, height),
                flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT)
            writer.write(stabilised)
            previous_gray = current_gray
            report_progress(progress, "Stabilising", processed + 1, total_frames, started)
    finally:
        video.release()
        writer.release()
    return output_path, {"reference": "first_frame", "frames": processed + 1}


def track_clip(path, model_path=None, tracker=TRACKER, frame_step=1,
               confidence=CLIP_CONFIDENCE, max_frames=None, progress=None,
               imgsz=CLIP_IMAGE_SIZE):
    """Track people with separate association for opposite direction classes."""
    if model_path is None:
        model_path = clip_model_for(path)
    model = YOLO(model_path)
    class_names = {key: model.names[key] for key in model.names}
    person_ids = [key for key in sorted(class_names) if "person" in class_names[key].lower()]
    if not person_ids:
        raise ValueError("The clip model has no class containing 'person'.")
    direction_ids = {
        name.lower(): class_id
        for class_id, name in class_names.items()
        if name.lower() in ("person_left", "person_right")
    }
    direction_available = len(direction_ids) == 2
    if direction_available:
        class_trackers = {
            "person_left": (direction_ids["person_left"], model),
            "person_right": (direction_ids["person_right"], YOLO(model_path)),
        }
        tracking_strategy = "separate_direction_class_trackers"
    else:
        class_trackers = {"all_person_classes": (person_ids, model)}
        tracking_strategy = "single_tracker_fallback"
    video = cv2.VideoCapture(path)
    fps = float(video.get(cv2.CAP_PROP_FPS))
    total_frames = int(video.get(cv2.CAP_PROP_FRAME_COUNT))
    limit = min(total_frames, max_frames) if max_frames is not None else total_frames
    started = time.time()
    report_progress(progress, "Tracking", 0, limit, started)
    tracks, detections = {}, 0
    global_ids = {}
    next_global_id = 1
    index = -1
    processed = 0
    while True:
        ok, frame = video.read()
        if not ok:
            break
        index += 1
        if index % frame_step:
            continue
        if max_frames is not None and processed >= max_frames:
            break
        processed += 1
        for label, (class_filter, tracker_model) in class_trackers.items():
            classes = [class_filter] if isinstance(class_filter, int) else class_filter
            result = tracker_model.track(
                frame, persist=True, tracker=tracker, conf=confidence,
                imgsz=imgsz, classes=classes, agnostic_nms=True,
                verbose=False,
            )[0]
            if result.boxes is None or result.boxes.id is None:
                continue
            detections += len(result.boxes.id)
            for box, raw_track_id, class_id, score in zip(
                result.boxes.xyxy.tolist(), result.boxes.id.int().tolist(),
                result.boxes.cls.int().tolist(), result.boxes.conf.tolist(),
            ):
                class_name = class_names[class_id]
                namespace = label if direction_available else "all_person_classes"
                key = (namespace, int(raw_track_id))
                if key not in global_ids:
                    global_ids[key] = next_global_id
                    next_global_id += 1
                track_id = global_ids[key]
                left, top, right, bottom = box
                tracks.setdefault(track_id, []).append({
                    "frame": index,
                    "seconds": round(index / fps, 3),
                    "box": [round(value, 1) for value in box],
                    "foot": [round((left + right) / 2, 1), round(bottom, 1)],
                    "class_name": class_name,
                    "confidence": round(float(score), 3),
                })
        report_progress(progress, "Tracking", processed, limit, started)
    video.release()
    return tracks, {
        "model_path": model_path,
        "tracker": tracker,
        "conf": confidence,
        "imgsz": imgsz,
        "direction_available": direction_available,
        "class_names": list(class_names.values()),
        "tracking_strategy": tracking_strategy,
        "tracker_instances": len(class_trackers),
        "processed_frames": processed,
        "total_frames": total_frames,
        "detections": detections,
    }


### From tracks to waits

A track is a list of observations, not a waiting event. The service uses a simple dwell-time rule:

1. **The bottom of the box must be inside or near the area.** The bottom-left,
   bottom-centre and bottom-right of the box are checked.
2. **The person must remain in the area.** The time from the first to the last
   in-area observation is the visible dwell time.
3. **The dwell must last.** A run shorter than `MIN_DWELL` is not reported.
4. **Brief gaps are allowed.** Missing detections or brief boundary flicker do not
   split one wait when the gap is no longer than `WAIT_GAP`.

A wait that touches the first or last frame of the clip is reported with
`at_least: true`. The true dwell is longer than the clip, and saying so is more
honest than reporting the visible part as the whole.


In [278]:
def final_label(observations):
    """Return the majority class label and its normalized vote margin."""
    label_counts = Counter(observation["class_name"] for observation in observations)
    ranked = label_counts.most_common(2)
    if not ranked:
        return "person", 0.0
    top_count = ranked[0][1]
    second_count = ranked[1][1] if len(ranked) > 1 else 0
    margin = (top_count - second_count) / max(1, sum(label_counts.values()))
    return ranked[0][0], round(margin, 3)


def find_waits(tracks, fps, clip_frames, area=WAITING_AREA,
               edge_tolerance=EDGE_TOLERANCE, min_dwell=MIN_DWELL,
               gap=WAIT_GAP, max_speed_px_s=MAX_WAIT_SPEED_PX_S):
    """Find people whose tracked feet remain in the ROI for min_dwell seconds.

    A brief missing-detection gap is joined to the same run when it is no longer
    than ``gap`` seconds. Speed is the median foot-point speed in each run.
    """
    waits = []
    last_clip_seconds = max(0.0, (clip_frames - 1) / fps)
    for track_id in sorted(tracks):
        observations = sorted(tracks[track_id], key=lambda item: item["seconds"])
        in_area = [o for o in observations if observation_in_area(o, area, edge_tolerance)]
        if not in_area:
            continue
        runs = []
        run = [in_area[0]]
        for observation in in_area[1:]:
            if observation["seconds"] - run[-1]["seconds"] > gap:
                runs.append(run)
                run = [observation]
            else:
                run.append(observation)
        runs.append(run)
        for run in runs:
            dwell = run[-1]["seconds"] - run[0]["seconds"]
            if dwell < min_dwell:
                continue
            speeds = []
            for previous, current in zip(run, run[1:]):
                elapsed = current["seconds"] - previous["seconds"]
                if elapsed > 0:
                    speeds.append(math.dist(previous["foot"], current["foot"]) / elapsed)
            median_speed = float(np.median(speeds)) if speeds else 0.0
            if median_speed > max_speed_px_s:
                continue
            label, _ = final_label(observations)
            waits.append({
                "track_id": int(track_id),
                "class_name": label,
                "first_frame": run[0]["frame"],
                "last_frame": run[-1]["frame"],
                "first_seconds": run[0]["seconds"],
                "last_seconds": run[-1]["seconds"],
                "dwell_seconds": round(dwell, 3),
                "at_least": bool(run[0]["seconds"] <= min_dwell or last_clip_seconds - run[-1]["seconds"] <= min_dwell),
                "max_radius_px": 0.0,
                "median_speed_px_s": round(median_speed, 3),
                "max_speed_px_s": round(max(speeds, default=0.0), 3),
            })
    waits.sort(key=lambda wait: (wait["first_seconds"], wait["track_id"]))
    return waits


### What the reply says

For a clip, `counts.waited` and `counts.crossed` count qualifying tracked IDs once.
The crossing total is also broken down by direction: `crossed_left_to_right` and
`crossed_right_to_left`. Direction comes from the model's `person_left` and
`person_right` class labels. A non-waiting track is counted regardless of whether
it enters a waiting-area polygon; tracks shorter than one second are discarded.

`waiting_people` lists each waiting track's `id` and `dwell_time`. The other
fields provide the status, confidence threshold, model version, whether trained
weights were used, clip duration, and processing time. The annotated video is
returned separately by the page.


In [279]:
def public_wait(wait):
    """The compact wait record returned to the caller."""
    return {
        "track_id": wait["track_id"],
        "class_name": wait["class_name"],
        "start_seconds": wait["first_seconds"],
        "end_seconds": wait["last_seconds"],
        "dwell_seconds": wait["dwell_seconds"],
        "at_least": wait["at_least"],
        "area": wait.get("area"),
        "max_radius_px": wait.get("max_radius_px", 0.0),
        "median_speed_px_s": wait.get("median_speed_px_s", 0.0),
        "max_speed_px_s": wait.get("max_speed_px_s", 0.0),
    }


def eligible_tracks(tracks, min_observations=MIN_TRACK_OBSERVATIONS,
                    min_duration=MIN_TRACK_DURATION_SECONDS):
    """Keep track IDs with enough detections spanning the minimum duration."""
    return {
        int(track_id): observations
        for track_id, observations in tracks.items()
        if len(observations) >= min_observations
        and max(item["seconds"] for item in observations)
        - min(item["seconds"] for item in observations) >= min_duration
    }


def summarize_clip_outcomes(eligible, waits):
    """Count each tracked ID once as waiting or as a directional crossing."""
    waiting_by_id = {}
    for wait in waits:
        track_id = int(wait["track_id"])
        person = waiting_by_id.setdefault(
            track_id, {"id": track_id, "wait_seconds": 0.0})
        person["wait_seconds"] += wait["dwell_seconds"]
    waiting_people = [
        {"id": person["id"], "dwell_time": round(person["wait_seconds"], 3)}
        for _, person in sorted(waiting_by_id.items())
    ]

    crossed_left_to_right = set()
    crossed_right_to_left = set()
    for track_id, observations in eligible.items():
        if track_id in waiting_by_id:
            continue
        label, _ = final_label(observations)
        if label == "person_left":
            crossed_left_to_right.add(track_id)
        elif label == "person_right":
            crossed_right_to_left.add(track_id)

    counts = {
        "waited": len(waiting_people),
        "crossed": len(crossed_left_to_right | crossed_right_to_left),
        "crossed_left_to_right": len(crossed_left_to_right),
        "crossed_right_to_left": len(crossed_right_to_left),
    }
    return counts, waiting_people


def build_wait_reply(uploaded, confidence=CLIP_CONFIDENCE,
                     min_dwell=MIN_DWELL, gap=WAIT_GAP, edge_tolerance=EDGE_TOLERANCE,
                     max_speed_px_s=MAX_WAIT_SPEED_PX_S,
                     max_frames=None, write_video=True, stabilise=False, progress=None,
                     area=None, areas=None, imgsz=CLIP_IMAGE_SIZE,
                     merge=True, tracker=TRACKER,
                     min_track_observations=MIN_TRACK_OBSERVATIONS,
                     min_track_duration=MIN_TRACK_DURATION_SECONDS):
    clip, error = validate_clip(uploaded)
    if error is not None:
        return error, None
    original_path = clip["path"]
    if areas is None:
        areas = {"left": area, "right": None}
    selected_areas, area_error, area_source = validate_waiting_areas(
        areas.get("left"), areas.get("right"), clip["width"], clip["height"], return_source=True)
    if area_error is not None:
        return {"status": 400, "error": "Invalid waiting area.", "detail": area_error}, None
    if area_source["left"] == "default_demo" and "demo1" not in os.path.basename(original_path).lower():
        print("WARNING: default_demo waiting area used for a clip that is not demo1.")
    model_path = clip_model_for(original_path)
    started = time.time()
    processing_path, processing_clip = original_path, clip
    if stabilise:
        try:
            processing_path, stabilisation_info = stabilise_clip(original_path, progress=progress)
            processing_clip, processed_error = validate_clip(processing_path)
            if processed_error is not None:
                raise RuntimeError(processed_error.get("detail", "Invalid stabilised video."))
        except Exception as problem:
            print("WARNING: stabilisation failed; using the original clip: %s" % problem)
            processing_path, processing_clip = original_path, clip
    try:
        tracks, tracking_info = track_clip(processing_path, model_path=model_path, confidence=confidence,
                                           max_frames=max_frames, progress=progress, imgsz=imgsz, tracker=tracker)
    except ValueError as problem:
        return {"status": 500, "error": "The clip model cannot track people.", "detail": str(problem)}, None
    except Exception as problem:
        return {"status": 500, "error": "The clip could not be tracked.", "detail": "%s: %s" % (type(problem).__name__, problem)}, None
    merger = globals().get("merge_tracklets")
    if merge and callable(merger):
        tracks, _ = merger(tracks, fps=processing_clip["fps"])
    eligible = eligible_tracks(
        tracks, min_observations=int(min_track_observations),
        min_duration=float(min_track_duration))
    waits = []
    frame_limit = min(processing_clip["frames"], max_frames) if max_frames is not None else processing_clip["frames"]
    for area_points in selected_areas.values():
        waits.extend(find_waits(
            eligible, processing_clip["fps"], frame_limit, area=area_points,
            edge_tolerance=edge_tolerance, min_dwell=min_dwell, gap=gap,
            max_speed_px_s=max_speed_px_s))
    waits.sort(key=lambda wait: (wait["first_seconds"], wait["track_id"]))
    counts, waiting_people = summarize_clip_outcomes(eligible, waits)

    video_path = None
    if write_video:
        try:
            video_path = annotate_waiting_clip(
                processing_path, tracks, waits, areas=selected_areas,
                edge_tolerance=edge_tolerance, max_frames=max_frames,
                progress=progress, started=started)
        except Exception as problem:
            print("The annotated video could not be written: %s" % problem)
    elapsed = round(time.time() - started, 3)
    reply = {
        "status": 200,
        "counts": counts,
        "waiting_people": waiting_people,
        "confidence_threshold": confidence,
        "model_version": SERVICE_VERSION,
        "trained_model": model_path == WEIGHTS,
        "clip_duration_seconds": processing_clip["duration_seconds"],
        "inference_seconds": elapsed,
    }
    if video_path is not None:
        with open(video_path.replace(".mp4", ".json"), "w", encoding="utf-8") as handle:
            json.dump(reply, handle, indent=2)
    print("waited: %d, crossed: %d, %.1fs" % (
        counts["waited"], counts["crossed"], elapsed))
    return reply, video_path


def analyse_clip(uploaded, confidence, min_dwell, gap, max_speed_px_s,
                 min_track_observations=MIN_TRACK_OBSERVATIONS,
                 min_track_duration=MIN_TRACK_DURATION_SECONDS,
                 edge_tolerance=EDGE_TOLERANCE, stabilise=False, progress=None,
                 area=None, areas=None, imgsz=CLIP_IMAGE_SIZE, tracker=TRACKER):
    """What the page calls for a clip."""
    reply, video_path = build_wait_reply(
        uploaded, confidence=confidence, min_dwell=min_dwell, gap=gap,
        max_speed_px_s=max_speed_px_s,
        edge_tolerance=edge_tolerance, stabilise=stabilise, progress=progress,
        area=area, areas=areas, imgsz=imgsz, tracker=tracker,
        min_track_observations=min_track_observations,
        min_track_duration=min_track_duration)
    return video_path, reply


---

## 5. The interface

Gradio turns functions into a page. It needs to be told what each function takes,
what it gives back, and what to call it. The page has one tab with two inputs:
the photograph counter from Section 3 and the waiting service from Section 4.
Either input can be sent on its own; the reply marks the missing half as skipped.

`share=True` is what produces the public link. Without it the page is reachable
only from inside this runtime, which means it has not been deployed to anybody.


The clip tab also has a stabilisation checkbox. It is enabled by default for
moving-camera footage. For a fixed camera, turn it off to avoid unnecessary
warping. Always inspect the stabilised result and recalibrate `WAITING_AREA`
against that output before relying on waiting-time results.


### Run order

Run the setup and import cells first, and confirm the import cell prints
`gradio: <version>`. If you edit the ROI functions, re-run that function cell
before rebuilding the page. Close an existing demo with `demo.close()` before
launching the rebuilt page again.


In [280]:
def annotate(image, detections):
    """Draw the kept boxes, so the reply can be checked by eye as well as read."""
    from PIL import ImageDraw

    marked = image.copy()
    draw = ImageDraw.Draw(marked)
    for detection in detections:
        left, top, right, bottom = detection["box"]
        colour = CLASS_COLOURS_RGB.get(
            detection["class_name"], (128, 128, 128))
        draw.rectangle([left, top, right, bottom], outline=colour, width=4)
        label = "%s %.2f" % (detection["class_name"], detection["confidence"])
        position = (left + 3, max(18, top - 16))
        draw.text(position, label, fill=(0, 0, 0), stroke_width=3,
                  stroke_fill=(0, 0, 0))
        draw.text(position, label, fill=(255, 255, 255), stroke_width=1,
                  stroke_fill=(0, 0, 0))
    return marked


def predict(uploaded, threshold):
    """What the page calls for a photograph. Returns the annotated image and reply."""
    reply, image = build_reply(uploaded, threshold)
    if image is None:
        return None, reply
    return annotate(image, reply["detections"]), reply


def analyse(photo, clip, photo_threshold, clip_confidence,
            clip_dwell, clip_gap, clip_speed, clip_track_observations,
            clip_track_duration, clip_buffer, clip_edge, clip_stabilise,
            roi_left, roi_right,
            progress=gr.Progress(track_tqdm=True)):
    """One page call: exactly one of a photograph or a clip."""
    if photo is not None and clip is not None:
        return None, None, {
            "status": 400,
            "error": "Send either a photograph or a clip, not both.",
            "detail": "Clear one of the two uploads and send again.",
        }

    if photo is None and clip is None:
        return None, None, {
            "status": 400,
            "error": "No input was sent.",
            "detail": "Add a photograph or a clip and send again.",
        }

    if photo is not None:
        annotated_image, reply = predict(photo, photo_threshold)
        return annotated_image, None, reply

    tracker = create_botsort_config(clip_buffer)
    annotated_video, reply = analyse_clip(
        clip, clip_confidence, clip_dwell, clip_gap, clip_speed,
        clip_track_observations, clip_track_duration, clip_edge,
        clip_stabilise, progress=progress,
        areas={"left": roi_left, "right": roi_right}, tracker=tracker)
    return None, annotated_video, reply


In [281]:
def toggle_help(opened):
    opened = not bool(opened)
    return opened, gr.update(visible=opened)


with gr.Blocks(title="People Counting Assistant") as demo:
    gr.Markdown(
        "## People Counting Assistant\n"
        "Send either a photograph or a clip, not both. The reply counts people "
        "in the photograph, or reports who waited inside the waiting area "
        "in the clip, and for how long."
    )

    how_button = gr.Button("How to use", variant="secondary")
    how_open = gr.State(False)
    how_text = gr.Markdown(
        """### How to use

1. Upload either a photograph or a video clip.
2. For a clip, click four points on the left preview to draw the main waiting area. The right area is optional.
3. Adjust the confidence threshold if detections are missing or noisy.
4. Set **Minimum time in ROI**: a person is reported as waiting after staying in the area for this long.
5. Set **Maximum waiting speed** in pixels per second. A person is reported only when their median tracked foot-point speed stays below this value.
6. Set **Allowed detection gap** to tolerate brief missed detections during one visit.
7. Set **Edge tolerance** if a person's feet are close to the ROI boundary.
8. The JSON counts each track seen for at least 1 second as a crossing, unless it met the waiting rule; waiters are listed with their ID and wait time.
9. Click **Count or analyse**, then inspect the annotated output and JSON reply.

For clips with both direction classes, left- and right-moving people are tracked in separate BoT-SORT instances so opposite classes cannot exchange IDs. A wrong or flickering class prediction may fragment a track; same-class people can still switch IDs. This runs two model/tracker passes per frame, so clip analysis may take longer. For moving-camera footage, enable stabilisation and inspect the annotated ROI before relying on dwell times.
""",
        visible=False,
    )
    how_button.click(toggle_help, [how_open], [how_open, how_text])

    with gr.Tab("Count or wait"):
        photo = gr.Image(type="pil", label="Walkway photograph")
        photo_threshold = gr.Slider(0.05, 0.95,
                                    value=CONFIDENCE_THRESHOLD, step=0.05,
                                    label="Photo confidence threshold")

        clip = gr.Video(label="Walkway clip")
        roi_source = gr.State(None)
        roi_left = gr.State([])
        roi_right = gr.State([])
        roi_preview_left = gr.Image(type="numpy", label="First frame - LEFT waiting area",
                                    interactive=False)
        roi_preview_right = gr.Image(type="numpy", label="First frame - RIGHT waiting area",
                                     interactive=False)
        roi_status_left = gr.Markdown("Upload a clip to show the first frame.")
        roi_status_right = gr.Markdown("Upload a clip to show the first frame.")
        clip.change(first_frame_preview, [clip],
                    [roi_preview_left, roi_preview_right, roi_source,
                     roi_left, roi_right, roi_status_left, roi_status_right])
        roi_preview_left.select(select_left_roi, [roi_source, roi_left],
                                [roi_preview_left, roi_left, roi_status_left])
        roi_preview_right.select(select_right_roi, [roi_source, roi_right],
                                 [roi_preview_right, roi_right, roi_status_right])
        clip_stabilise = gr.Checkbox(
            value=False,
            label="Stabilise moving-camera footage before applying the ROI"
        )
        with gr.Row():
            clip_confidence = gr.Slider(0.05, 0.95, value=CLIP_CONFIDENCE,
                                        step=0.05,
                                        label="Clip confidence threshold")
            clip_dwell = gr.Slider(0.5, 15.0, value=MIN_DWELL, step=0.5,
                                   label="Minimum time in ROI, s")
            clip_gap = gr.Slider(0.0, 3.0, value=WAIT_GAP, step=0.25,
                                 label="Allowed detection gap, s")
            clip_speed = gr.Slider(0.0, 200.0, value=MAX_WAIT_SPEED_PX_S, step=1.0,
                                   label="Maximum waiting speed, px/s")
            clip_track_observations = gr.Slider(1, 30, value=MIN_TRACK_OBSERVATIONS, step=1,
                                               label="Minimum track observations")
            clip_track_duration = gr.Slider(0.5, 15.0, value=MIN_TRACK_DURATION_SECONDS, step=0.5,
                                            label="Minimum track duration, s")
        clip_buffer = gr.Slider(1, 180, value=TRACKER_BUFFER, step=1,
                                    label="Tracker memory buffer, frames")
        clip_edge = gr.Slider(0, 60, value=EDGE_TOLERANCE, step=1,
                              label="Edge tolerance, px")

        run_button = gr.Button("Count or analyse", variant="primary")
        photo_image = gr.Image(label="Photo detections")
        clip_video = gr.Video(label="Annotated clip")
        reply = gr.JSON(label="Reply")
        run_button.click(analyse,
                         [photo, clip, photo_threshold, clip_confidence,
                          clip_dwell, clip_gap, clip_speed, clip_track_observations,
                          clip_track_duration, clip_buffer, clip_edge, clip_stabilise,
                          roi_left, roi_right],
                         [photo_image, clip_video, reply])

    demo.launch(share=True)


* Running on local URL:  http://127.0.0.1:7874
* Running on public URL: https://44c9022939f8f3caff.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Two things worth knowing about the public link before it goes in a report.

- It lives for 72 hours and dies the moment this runtime stops. It is a demo
  link, not a deployment.
- Anyone holding it can send photographs and clips to it. There is no account and
  no key.

Send the link to another team and have them send a photograph taken on a
different phone at a different time of day. A service that has only ever been
called by the person who built it has not been tested. The clip half is worth
the same test with a clip from a different phone, if the camera position matches
the selected waiting areas.


For moving-camera footage, enable the stabilisation checkbox before analysing
the clip. Stabilisation is an image-processing correction, not a guarantee:
large viewpoint changes and parallax can still move the physical area relative
to the image. Inspect the annotated result before treating the dwell times as
valid.


---

## 6. Testing the error conditions

A deployment should explain invalid input instead of exposing a traceback. These tests call the service functions directly, so they exercise the same validation that would run inside the Gradio page or a future API.

The image cases cover missing, empty, unreadable, oversized, and valid-but-empty input. The clip cases cover missing, empty, unreadable, too-long, and invalid waiting-area input. The clip checks use the notebook's existing `CLIP_PATH`, `MAX_CLIP_FRAMES`, and ROI validation, so the Botsort configuration is not changed.

If no clip is configured, clip-specific tests are skipped. The temporary frame-limit test restores `MAX_CLIP_FRAMES` even if validation raises an exception.


In [282]:
import tempfile

error_test_dir = tempfile.mkdtemp(prefix="lesson4_errors_")
text_image = os.path.join(error_test_dir, "not_an_image.txt")
with open(text_image, "w", encoding="utf-8") as handle:
    handle.write("this is not a photograph")

empty_image = os.path.join(error_test_dir, "empty.jpg")
open(empty_image, "wb").close()

huge_image = Image.new("RGB", (9000, 7000), (200, 200, 200))
huge_image_path = os.path.join(error_test_dir, "huge.jpg")
huge_image.save(huge_image_path, quality=30)

nobody_image = Image.new("RGB", (640, 480), (214, 210, 202))
nobody_image_path = os.path.join(error_test_dir, "nobody.jpg")
nobody_image.save(nobody_image_path)

image_cases = [
    ("nothing sent", None),
    ("a text file", text_image),
    ("an empty file", empty_image),
    ("a very large image", huge_image_path),
    ("an image with nobody in it", nobody_image_path),
]

image_rows = []
for label, payload in image_cases:
    reply, _ = build_reply(payload)
    summary = reply.get("error") or ("counts " + str(reply.get("counts")))
    image_rows.append((label, reply["status"], summary))

print("%-30s %-7s %s" % ("image case", "status", "what came back"))
for label, status, summary in image_rows:
    print("%-30s %-7s %s" % (label, status, summary))


counted {'person_left': 0, 'person_right': 0} in 0.182s on a 640x480 image
image case                     status  what came back
nothing sent                   400     No image was sent.
a text file                    415     That file is not a readable image.
an empty file                  400     The upload was empty.
a very large image             413     The image is too large to process.
an image with nobody in it     200     counts {'person_left': 0, 'person_right': 0}


In [283]:
test_tracks = {
    1: [{"class_name": "person_left", "seconds": 0.0},
        {"class_name": "person_left", "seconds": 1.2}],
    2: [{"class_name": "person_right", "seconds": 0.0},
        {"class_name": "person_right", "seconds": 1.5}],
    3: [{"class_name": "person_right", "seconds": 0.0},
        {"class_name": "person_right", "seconds": 1.2}],
    4: [{"class_name": "person_left", "seconds": 0.0},
        {"class_name": "person_left", "seconds": 0.8}],
}
test_eligible = eligible_tracks(test_tracks, min_observations=2, min_duration=1.0)
test_waits = [
    {"track_id": 2, "dwell_seconds": 2.0},
    {"track_id": 2, "dwell_seconds": 1.5},
]
test_counts, test_waiting = summarize_clip_outcomes(test_eligible, test_waits)
assert test_counts == {
    "waited": 1,
    "crossed": 2,
    "crossed_left_to_right": 1,
    "crossed_right_to_left": 1,
}
assert test_waiting == [{"id": 2, "dwell_time": 3.5}]
assert 4 not in test_eligible
print("clip outcome checks passed: duration filter, unique counts, directions, wait IDs")

clip_test_dir = tempfile.mkdtemp(prefix="lesson4_clip_errors_")
text_clip = os.path.join(clip_test_dir, "not_a_video.txt")
with open(text_clip, "w", encoding="utf-8") as handle:
    handle.write("this is not a video")

empty_clip = os.path.join(clip_test_dir, "empty.mp4")
open(empty_clip, "wb").close()

clip_cases = [
    ("nothing sent", None),
    ("a text file", text_clip),
    ("an empty file", empty_clip),
]

clip_rows = []
for label, payload in clip_cases:
    reply, _ = build_wait_reply(payload, write_video=False)
    summary = reply.get("error") or ("waiting " + str(reply.get("waiting_count")))
    clip_rows.append((label, reply["status"], summary))

if CLIP_PATH is not None:
    original_limit = MAX_CLIP_FRAMES
    try:
        MAX_CLIP_FRAMES = 1
        long_reply, _ = build_wait_reply(CLIP_PATH, write_video=False)
    finally:
        MAX_CLIP_FRAMES = original_limit
    clip_rows.append(("clip longer than MAX_CLIP_FRAMES", long_reply["status"], long_reply.get("error", "ok")))

    roi_cases = [
        ("ROI with 3 points", [[0, 0], [10, 0], [10, 10]]),
        ("ROI point outside frame", [[0, 0], [2000, 0], [10, 10], [0, 10]]),
        ("zero-area ROI", [[10, 10], [20, 20], [30, 30], [40, 40]]),
    ]
    for label, roi in roi_cases:
        reply, _ = build_wait_reply(
            CLIP_PATH,
            areas={"left": roi, "right": None},
            write_video=False,
        )
        clip_rows.append((label, reply["status"], reply.get("error", "ok")))
else:
    print("No clip is configured; clip-specific tests were skipped.")

print("\n%-30s %-7s %s" % ("clip case", "status", "what came back"))
for label, status, summary in clip_rows:
    print("%-30s %-7s %s" % (label, status, summary))


clip outcome checks passed: duration filter, unique counts, directions, wait IDs

clip case                      status  what came back
nothing sent                   400     No clip was sent.
a text file                    415     That file is not a readable video.
an empty file                  400     The upload was empty.
clip longer than MAX_CLIP_FRAMES 413     The clip is too long to process.
ROI with 3 points              400     Invalid waiting area.
ROI point outside frame        400     Invalid waiting area.
zero-area ROI                  400     Invalid waiting area.


---

## 7. Cold and warm response time

A cold start is what somebody waits through when they are the first to arrive
after the service has been sitting idle, or has just been deployed.

It cannot be measured from in here. A cold start is a whole process starting:
importing the libraries, setting up the GPU, reading the weights off a disk that
has never read them before, and only then answering. This notebook paid every one
of those in Section 2 and will not pay them again, so timing another load in this
process measures a warm service and calls it cold.

The warm number is measured here, where the conditions are right. The cold one is
measured by starting a new Python process that does nothing but import, load and
answer once, and timing the whole thing from outside, which is what a host does
when it starts a service after an idle period.

The clip half is different in one respect: its time grows with the number of
frames. It is measured in seconds rather than milliseconds. Its reply carries
`seconds`, which is the number to record.


In [284]:
# Warm, measured here, where this process is exactly the warm service.
sample = Image.new("RGB", (640, 480), (214, 210, 202))

warm = []
for attempt in range(5):
    started = time.time()
    count_people(sample, CONFIDENCE_THRESHOLD)
    warm.append(time.time() - started)

warm_average = sum(warm) / len(warm)
print("warm inference, average : %.3f s" % warm_average)

warm inference, average : 0.060 s


In [285]:
import subprocess
import sys

# A separate process, so nothing this notebook has already done counts. It
# imports, loads the weights and answers one request, and reports where its own
# time went. The figure that matters is the total measured from out here, because
# that is what the caller waits through.
weights = os.path.abspath(MODEL_PATH) if os.path.exists(MODEL_PATH) else MODEL_PATH
probe_dir = tempfile.mkdtemp()
probe_path = os.path.join(probe_dir, "cold_start_probe.py")

with open(probe_path, "w", encoding="utf-8") as handle:
    print("import sys, time", file=handle)
    print("launch = float(sys.argv[1])", file=handle)
    print("start = time.time()", file=handle)
    print("from ultralytics import YOLO", file=handle)
    print("from PIL import Image", file=handle)
    print("imported = time.time()", file=handle)
    print("model = YOLO(%r)" % weights, file=handle)
    print("loaded = time.time()", file=handle)
    print("model.predict(Image.new('RGB', (640, 480)), conf=0.01, verbose=False)",
          file=handle)
    print("answered = time.time()", file=handle)
    print("print('  importing the libraries : %.3f s' % (imported - start))", file=handle)
    print("print('  reading the weights     : %.3f s' % (loaded - imported))", file=handle)
    print("print('  answering the first time: %.3f s' % (answered - loaded))", file=handle)
    print("print('ANSWERED %.3f' % (answered - launch))", file=handle)

started = time.time()
probe = subprocess.run([sys.executable, probe_path, repr(started)],
                       capture_output=True, text=True)
process_lifetime = time.time() - started

# The caller waits from the moment the process is launched until it answers. What
# happens after that, releasing the GPU and shutting down, can take seconds and
# nobody is waiting for it, so timing the whole subprocess overstates the wait.
cold_total = process_lifetime
report = []
for line in probe.stdout.splitlines():
    if not line.strip():
        continue
    if line.startswith("ANSWERED "):
        cold_total = float(line.split()[1])
    else:
        report.append(line)

if probe.returncode != 0:
    print("The probe did not run. Its last words:")
    print(probe.stderr.strip()[-600:])
else:
    print("inside a process that has just started:")
    for line in report:
        print(line)
    print("")
    print("somebody arriving at a cold service waits : %.3f s" % cold_total)
    print("somebody arriving at a warm one waits     : %.3f s" % warm_average)
    print("cold is %.0f times the warm figure" % (cold_total / warm_average))
    print("")
    print("the probe process lived for %.3f s in total. The difference is it"
          % process_lifetime)
    print("shutting down and releasing the GPU, which no caller waits for.")

inside a process that has just started:
  importing the libraries : 3.403 s
  reading the weights     : 0.138 s
  answering the first time: 3.303 s

somebody arriving at a cold service waits : 6.902 s
somebody arriving at a warm one waits     : 0.060 s
cold is 115 times the warm figure

the probe process lived for 8.245 s in total. The difference is it
shutting down and releasing the GPU, which no caller waits for.


Read where the time went, not only the total. The answering is milliseconds and
everything before it is not, which is the whole shape of the problem: this
service is slow to start and fast to answer.

None of that is a bug, and it is worth knowing which, because a team that does
not know it will spend lab time debugging a service that works.

It also decides something real about where this gets deployed. A host that shuts
an idle service down to save money pays that load cost again on the very next
call, so the first person to use it after a quiet period waits through all of it
while everybody after them waits milliseconds. A host that keeps it warm charges
for the idle time instead. Neither is free, and which one is right depends on how
often the walkway is actually looked at.

The scrollback above this cell is the log: every call printed a line, and reading
that log is how a failure gets diagnosed once the caller is somebody else.

The clip half is the exception to "fast to answer": it reads every frame, so its
time grows with the clip. The reply reports that time, and the log prints one line
per clip. A 38 second clip will always look slow next to a photograph; that is the
work, not a bug.


In [286]:
#demo.close()
#print("The share link is closed.")